# SEVIRI SSL: supplementary radiometric probes

Results of frozen-encoder forecasting, ridge regression and temporal-intervention experiments on Small encoders pretrained with **4,096 or 8,192 clips**. The fitting observations are the original pretraining clips; validation uses 512 clips from 17 September 2020. The sections below specify outputs, fitting procedures and evaluation metrics.

Related results: [SSL training](study_seviri_ssl_training.ipynb) and [CMA/CT/RDT probing](study_seviri_cloud_mask_probe.ipynb).

Execute cells in order with **Python (deep_convection)**. Figures and CSV exports are written to `results/radiometric_probes_study_20261110/`.

In [ ]:
from pathlib import Path
import importlib, json, sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/emma_gpm").is_dir())
sys.path.insert(0, str(ROOT / "src"))
SERIES = ROOT / "results/ssl_scaling_20261025"
SERIES_PATHS = [SERIES, ROOT / "results/ssl_extension_20261026", ROOT / "results/ssl_base_32768_20261026"]
LEARNING_SERIES_PATHS = SERIES_PATHS + [ROOT / "results/ssl_factorial_20261028"]
OUTPUT = ROOT / "results/radiometric_probes_study_20261110"
OUTPUT.mkdir(exist_ok=True)
USE_WIDGETS = False
SHOW_TABLES = False
if "monitor" in globals():
    monitor.close()
from emma_gpm import ssl_study, ssl_dashboard
study = importlib.reload(ssl_study)
views = importlib.reload(ssl_dashboard)
loaded_studies = [study.load_study(path) for path in SERIES_PATHS if (path / "scaling_plan.json").exists()]
SNAPSHOT = study.merge_studies(*loaded_studies)
learning_studies = loaded_studies + [study.load_study(path) for path in LEARNING_SERIES_PATHS
                                   if path not in SERIES_PATHS and (path / "scaling_plan.json").exists()]
LEARNING_SNAPSHOT = study.merge_studies(*learning_studies)
GRID_COMPLETION = ROOT / "results/ssl_grid_completion_20261103"
if (GRID_COMPLETION / "scaling_plan.json").exists():
    LEARNING_SNAPSHOT = study.add_completed_experiments(LEARNING_SNAPSHOT, study.load_study(GRID_COMPLETION))
SNAPSHOT = LEARNING_SNAPSHOT
GEOGRAPHIC_SERIES = next((ROOT / name for name in (
    "results/ssl_131072_verified_20261102", "results/ssl_131072_20261102")
    if (ROOT / name / "scaling_plan.json").exists()), ROOT / "results/ssl_131072_verified_20261102")
GEOGRAPHIC_SNAPSHOT = (study.load_study(GEOGRAPHIC_SERIES)
                       if (GEOGRAPHIC_SERIES / "scaling_plan.json").exists() else None)

LEONARDO_TRAINING = Path("/data1/deep_convection_results/leonardo_base_future_131072_20261108/training")
if GEOGRAPHIC_SNAPSHOT and (LEONARDO_TRAINING / "progress.json").exists():
    GEOGRAPHIC_SNAPSHOT = study.attach_distributed_training(GEOGRAPHIC_SNAPSHOT, LEONARDO_TRAINING)

def table(frame):
    if not SHOW_TABLES:
        return
    if frame.empty:
        print("This comparison has no saved results in the selected snapshot.")
        return
    formats = {column: "{:.0f}" if pd.api.types.is_integer_dtype(frame[column]) else "{:.2f}"
               for column in frame.select_dtypes(include="number").columns}
    for column in frame.columns:
        if any(term in column.lower() for term in ("epoch", "clips", "updates", "models", "budget")):
            if pd.api.types.is_numeric_dtype(frame[column]):
                formats[column] = "{:.0f}"
        if "(%)" in column or column in ("Mean BT (K)", "BT standard deviation (K)"):
            formats[column] = "{:.1f}"
    display(frame.style.format(formats, na_rep="—").hide(axis="index"))

def figure(fig, filename):
    if fig is not None:
        fig.savefig(OUTPUT / filename, dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)

display(Markdown(f"**Snapshot:** {LEARNING_SNAPSHOT['captured_at'][:19].replace('T', ' ')} Europe/Paris · "
                 f"**Learning-curve series:** {len(learning_studies)} · **Figures and CSV files:** `{OUTPUT.relative_to(ROOT)}`"))

## Forecasting probes: head, data and evaluation

**Current output and target.** Each forecasting probe receives the ten historical frames and outputs **two complete future images**, each with seven channels and a 224 × 224 pixel grid. The output array per clip has shape **(2, 7, 224, 224)**. The corresponding targets are the observed SEVIRI images at **+15 and +30 minutes**, taken from stored clip frames 12 and 15. Training minimizes squared temperature error using the established global normalization; the displayed predictions and metrics are in kelvin.

The common forecasting probes are available for **Small at 4,096 and 8,192 clips**. For each objective, the final pretrained encoder is loaded and its parameters are frozen. A newly initialized spatial forecasting head is then fitted using all historical tokens. The random reference uses the same architecture with its original random encoder weights frozen. Encoder weight checks saved after training confirm that all ten probe encoders remained unchanged.

The fitted head has a **1 × 1 convolution, a 3 × 3 convolution and a transposed convolution**, with GELU activations between the first layers and hidden width 192. It combines the five temporal feature vectors at each spatial patch and reconstructs the two seven-channel future images. For Small, this head has about **1.4 million trainable parameters**. It is a nonlinear spatial decoder. All five encoders at a given size receive exactly the same head architecture, initialization, optimization settings and training budget: **150 epochs**, effective batch 32 and learning rate 0.00015. Its prediction target is the observed image at +15 and +30 minutes.

**Head-fitting observations are the same training clips used for that encoder's pretraining.** The 4,096-clip probes use 12 and 13 September 2020; the 8,192-clip probes also use 15 September. Every probe is evaluated on the same **512 clips from 17 September**, which are temporally separated from those training observations. Within a size, the five heads therefore have identical fitting and validation data. The forecast targets for the heads are computed directly from those observed future frames.

**Radiometric ridge outputs and targets.** Each ridge probe outputs **14 scalar values per clip**. Seven targets are the historical mean temperatures of the seven channels. The other seven are historical IR108 variability, historical net temperature change, the final historical frame's mean spatial-gradient magnitude, mean future IR108 temperature change at each of the two lead times, and mean absolute future advection residual at each lead time. These targets are calculated from the observed images. The ridge fits a linear mapping from frozen summary embeddings and uses the same training/validation split. The later figure separates historical observables from future observables.

In the following plots, the name in parentheses identifies the frozen encoder. **RMSE reduction vs Random** compares a pretrained encoder with the random-encoder probe at the same training size and epoch; positive values indicate improved prediction. In these forecasting-probe learning curves, colour identifies the frozen encoder used by the probe and line style identifies the training-set size. All learning curves use optimizer updates on the horizontal axis and evaluate the same future-image objective. The probe figures cover the two registered Small datasets; detailed coverage is exported as CSV.

In [ ]:
figure(study.learning_figure(SNAPSHOT, "frozen_common_head"), "frozen_probe_learning.png")
probe_selection, probe_coverage = study.select_budget(SNAPSHOT, "frozen_common_head")
probe_scores = study.metrics_frame(SNAPSHOT, probe_selection)
probe_summary = study.forecast_summary(probe_scores)
table(probe_coverage)
probe_coverage.to_csv(OUTPUT / "frozen_probe_coverage.csv", index=False)
table(probe_summary)
if not probe_scores.empty:
    figure(study.overview_figure(probe_scores), "frozen_probe_overview.png")
probe_budgets = study.size_comparison(SNAPSHOT, "frozen_common_head")
probe_budgets.to_csv(OUTPUT / "frozen_probe_matched_budgets.csv", index=False)
probe_summary.to_csv(OUTPUT / "frozen_probe_summary.csv", index=False)

probe_reference = study.probe_reference_summary(probe_summary)
figure(study.probe_reference_figure(probe_reference), "frozen_probe_vs_random.png")
table(probe_reference)
probe_reference.to_csv(OUTPUT / "frozen_probe_vs_random.csv", index=False)
for size, rows in probe_reference.groupby("Training clips"):
    learned = rows[rows.Model != "Probe (Random)"].sort_values("RMSE (K)")
    if learned.empty:
        continue
    better = learned[learned["RMSE reduction vs Random (K)"] > 0]
    higher = learned[learned["RMSE reduction vs Random (K)"] <= 0]
    statements = []
    if not better.empty:
        values = "; ".join(f'{row.Model}: {row["RMSE reduction vs Random (K)"]:.2f} K lower RMSE' for _, row in better.iterrows())
        statements.append(f"At {size:,} clips, the pretrained encoders improving on the Random reference are {values}.")
    if not higher.empty:
        values = "; ".join(f'{row.Model}: {-row["RMSE reduction vs Random (K)"]:.2f} K higher RMSE' for _, row in higher.iterrows())
        statements.append(f"The other pretrained encoders give {values}.")
    display(Markdown(" ".join(statements)))

## Temporal interventions and representation analysis

The saved representation analyses cover the 4,096- and 8,192-clip Small experiments. Their availability is recorded alongside the later forecasting runs in the exported diagnostic inventory.

### Response to changes in the input history

The temporal comparisons change the encoder input while keeping model weights and forecast targets fixed. **Chronological** uses the ten historical frames in their observed order. **Shuffled history** permutes the first nine frames reproducibly and retains the final observed frame in its original position. **Last frame repeated** uses ten copies of that final frame.

Persistence and Advection are calculated from the original history for every comparison. Residual also retains its original advective component. The resulting error changes measure how the neural prediction responds to historical content and ordering under a fixed transport reference. Both temporal information and the change in input distribution contribute to this response.

The embedding comparison adds **Advection-only history**, a synthetic sequence formed by repeatedly translating the first historical image with the displacement estimated from the last two real historical IR108 images. Positions outside the crop retain the first image's value at the corresponding pixel. The saved diagnostics include the fraction of pixels supported by the translation.

### Representation summaries and radiometric probes

The representation analysis first averages the encoder features over the spatial grid, producing one 384-component vector at each of the five temporal positions. It concatenates the temporal mean of these vectors with their signed linear temporal trend, giving a 768-component embedding. The trend retains the direction of feature changes across the history. Ridge probes and embedding-distance comparisons use this summary; the common forecasting probes use the full spatial feature grid.

The **Embedding sensitivity** plot shows mean cosine distance between the original and altered embeddings over the 512 validation clips. Larger distances indicate a larger change in feature-vector direction under the selected input intervention. Forecasting and radiometric probes assess the predictive information available in those features.

The ridge analysis uses the same regularization strength of 10 for all five encoders. Embedding components are centred and scaled using training-set statistics, and regression coefficients are fitted on the training clips before evaluation on validation.

There are 14 radiometric targets. Seven are historical brightness-temperature means, one per channel, averaged over the ten input frames and all crop pixels. The remaining seven describe IR108 dynamics and structure: historical variability; the mean temperature change across the 45-minute history, expressed as a rate in K per 15 minutes; the final input's mean absolute spatial gradient, in K/pixel; the mean temperature change from the final input to each of the two future images; and the mean absolute difference between the observed and advected image at each future lead time. In the ridge plots, **residual amplitude** denotes this last quantity in kelvin.

The dot plots focus on these seven dynamic and structural targets and report validation **R²**, the coefficient of determination. A value of 1 indicates exact predictions, 0 matches the squared error obtained by predicting the validation mean, and negative values indicate greater error than that reference. The saved results also include MAE and the error obtained by predicting the training-target mean.

### Availability in the current series

The exported diagnostic inventory identifies the analysis products already saved for each dataset size. The supervisor runs this analysis after the scheduled training runs. Existing current-series products are displayed when available; the report retains their model and intervention labels.

The temporal plots compare global RMSE under each intervention with chronological history. Exact changes for every model and training size are exported in `temporal_rmse_changes.csv`, using the same validation observations. The radiometric interpretation below focuses on predicting the signed mean temperature change of the whole crop and the amplitude of the advection residual; these are distinct forecast targets.

In [ ]:
availability = study.diagnostic_availability(SNAPSHOT)
table(availability)
availability.to_csv(OUTPUT / "diagnostic_availability.csv", index=False)
for case, data in SNAPSHOT["cases"].items():
    if (data["queue"] / "analysis/temporal_ablations.csv").exists():
        display(Markdown(f"**Representation diagnostics · {data['train_samples']:,} training clips**"))
        views.representation_diagnostics(data["queue"], show_tables=SHOW_TABLES)

temporal_rows = []
for case, data in SNAPSHOT["cases"].items():
    analysis = data["queue"] / "analysis"
    if study.read_json(analysis / "progress.json").get("status") != "completed":
        continue
    temporal = pd.read_csv(analysis / "temporal_ablations.csv")
    pivot = temporal.pivot(index="run", columns="ablation", values="rmse_k")
    for model, values in pivot.iterrows():
        temporal_rows.append({"Training clips": data["train_samples"], "Model": views.LABELS[model],
            "Chronological RMSE (K)": values["chronological"],
            "Shuffled-history increase (K)": values["shuffle"] - values["chronological"],
            "Repeated-last-frame increase (K)": values["last_frame"] - values["chronological"]})
    ridge = pd.read_csv(analysis / "radiometric_probes.csv")
    for minutes in (15, 30):
        target = f"future_ir108_mean_change_{minutes}min_k"
        rows = ridge[ridge.descriptor == target]
        best = rows.loc[rows.r2.idxmax()]
        if (rows.r2 < 0).all():
            display(Markdown(f"With **{data['train_samples']:,} clips**, all five ridge probes have negative validation R² "
                f"for the crop's signed mean IR108 change at +{minutes} minutes. "
                f"The highest R² is {best.r2:.2f}, obtained by {views.LABELS[best.run]}. "
                "Their squared errors exceed the reference obtained from the validation-target mean."))
    residual = ridge[ridge.descriptor.str.startswith("future_ir108_advection_residual_mae_")]
    winners = [group.loc[group.r2.idxmax()] for _, group in residual.groupby("descriptor")]
    if len(winners) == 2 and all(row.run == "probe_random" for row in winners):
        values = ", ".join(f"+{15 if '15min' in row.descriptor else 30} minutes: R² {row.r2:.2f}" for row in winners)
        display(Markdown(f"At the same training size, **Probe (Random)** has the highest ridge R² "
            f"for future residual amplitude at both lead times ({values}). "
            "The common forecasting head and these ridge probes therefore reveal different aspects of the representations: "
            "prediction of full spatial images and linear recovery of crop-level quantities."))
temporal_comparison = pd.DataFrame(temporal_rows)
table(temporal_comparison)
temporal_comparison.to_csv(OUTPUT / "temporal_rmse_changes.csv", index=False)